## Grokking Hacking 1

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install transformer_lens==2.11.0

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import os
import sys
from functools import partial
from pathlib import Path

import einops
import numpy as np
import torch as t
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from jaxtyping import Float
from torch import Tensor
from tqdm import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.utils import to_numpy
import matplotlib.pyplot as plt

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part52_grokking_and_modular_arithmetic"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

grokking_root = section_dir / "Grokking"
saved_runs_root = grokking_root / "saved_runs"

import part52_grokking_and_modular_arithmetic.tests as tests
import part52_grokking_and_modular_arithmetic.utils as utils

device = t.device("cuda" if t.cuda.is_available() else "cpu")

t.set_grad_enabled(False)

MAIN = __name__ == "__main__"

In [ ]:
p = 113

cfg = HookedTransformerConfig(
    n_layers=1,
    d_vocab=p + 1,
    d_model=128,
    d_mlp=4 * 128,
    n_heads=4,
    d_head=128 // 4,
    n_ctx=3,
    act_fn="relu",
    normalization_type=None,
    device=device,
)

model = HookedTransformer(cfg)

In [ ]:
if not grokking_root.exists():
    os.system(
        f'git clone https://github.com/neelnanda-io/Grokking.git "{grokking_root.as_posix()}"'
    )
    assert grokking_root.exists()
    os.mkdir(grokking_root / "large_files")

In [ ]:
REPO_ID = "callummcdougall/grokking_full_run_data"
FILENAME = "full_run_data.pth"

local_dir = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

full_run_data = t.load(local_dir, weights_only=True)
state_dict = full_run_data["state_dicts"][400]

model = utils.load_in_state_dict(model, state_dict)

In [ ]:
# Helper variables
W_O = model.W_O[0]
W_K = model.W_K[0]
W_Q = model.W_Q[0]
W_V = model.W_V[0]
W_in = model.W_in[0]
W_out = model.W_out[0]
W_pos = model.W_pos
W_E = model.W_E[:-1]
final_pos_resid_initial = model.W_E[-1] + W_pos[2]
W_U = model.W_U[:, :-1]

print("W_O  ", tuple(W_O.shape))
print("W_K  ", tuple(W_K.shape))
print("W_Q  ", tuple(W_Q.shape))
print("W_V  ", tuple(W_V.shape))
print("W_in ", tuple(W_in.shape))
print("W_out", tuple(W_out.shape))
print("W_pos", tuple(W_pos.shape))
print("W_E  ", tuple(W_E.shape))
print("W_U  ", tuple(W_U.shape))

In [ ]:
# W_O

In [ ]:
# Get all data and labels, and cache activations
all_data = t.tensor([(i, j, p) for i in range(p) for j in range(p)]).to(device)
labels = t.tensor([utils.target_fn(i, j) for i, j, _ in all_data]).to(device)
original_logits, cache = model.run_with_cache(all_data)

# Final position only, also remove the logits for `=`
original_logits = original_logits[:, -1, :-1]

# Get cross entropy loss
original_loss = utils.cross_entropy_high_precision(original_logits, labels)
print(f"Original loss: {original_loss.item():.3e}")

Good point to start poking around

In [ ]:
cache.keys()

In [ ]:
for k in cache.keys():
    print(k, cache[k].shape)

In [ ]:
input_1=t.tensor([[60, 60, 113]]).to(device)
out_1=model(input_1)
out_1[0, -1, :-1].argmax()

### Ok, let's start with weights, then do activations.

In [ ]:
plt.imshow(W_E.detach().cpu().numpy())

In [ ]:
plt.imshow(W_U.detach().cpu().numpy())

In [ ]:
W_E[0,:].shape

In [ ]:
plt.plot(W_E[100,:].detach().cpu().numpy())

In [ ]:
plt.plot(W_E[:,2].detach().cpu().numpy())

In [ ]:
W_E[:,20].shape

In [ ]:
plt.plot(W_U[:,2].detach().cpu().numpy())

In [ ]:
plt.plot(W_U[2,:].detach().cpu().numpy())

Ok sure so like maybe some structure, but nothing I would pick upon yet!

In [ ]:
plt.imshow(W_in.detach().cpu().numpy())

In [ ]:
plt.imshow(W_out.detach().cpu().numpy().T)

In [ ]:
plt.imshow(W_O[0].detach().cpu().numpy().T)

In [ ]:
plt.imshow(W_O[1].detach().cpu().numpy().T)

In [ ]:
plt.imshow(W_K[0].detach().cpu().numpy().T)

In [ ]:
plt.imshow(W_Q[0].detach().cpu().numpy().T)

Ok the key and query matrices are kinda interesting! Maybe they fit together in an interesting way?

In [ ]:
plt.imshow(W_V[0].detach().cpu().numpy().T)

In [ ]:
plt.imshow(W_K[1].detach().cpu().numpy().T)

In [ ]:
plt.imshow(W_K[2].detach().cpu().numpy().T)

Ok that one looks very Fourier transformy! Could these be cool as a 3d surface? Or too jaged? We'll see. 

In [ ]:
plt.imshow(W_K[3].detach().cpu().numpy().T)

In [ ]:
plt.imshow(W_Q[1].detach().cpu().numpy().T)

In [ ]:
plt.imshow(W_Q[2].detach().cpu().numpy().T)

In [ ]:
plt.imshow(W_Q[3].detach().cpu().numpy().T)

- Ok cool so I would say like hints of structure in query and key matrix weights - why these!?
- Interesting! 

## Ok now, activations

In [ ]:
from ipywidgets import interact

In [ ]:
def viz_embeddings(x=0, y=0):
    input_1=t.tensor([[x, y, 113]]).to(device)
    out_1, cache=model.run_with_cache(input_1)    
    plt.plot(cache['hook_embed'][0].detach().cpu().numpy()[:2].T);
    plt.show()

In [ ]:
# interact(viz_embeddings) #, x=(0, 1, 113), y=(0, 1, 113))
interact(viz_embeddings, x=(0, 113, 1), y=(0, 113, 1))

In [ ]:
input_1=t.tensor([[51, 71, 113]]).to(device)
out_1, cache=model.run_with_cache(input_1)
out_1[0, -1, :-1].argmax()

In [ ]:
plt.imshow(cache['blocks.0.attn.hook_q'][0, 0].detach().cpu().numpy())

In [ ]:
cache['blocks.0.attn.hook_q'].shape

In [ ]:
def viz_attn_1(x=0, y=0):
    input_1=t.tensor([[x, y, 113]]).to(device)
    out_1, cache=model.run_with_cache(input_1)    
    fig=plt.figure(0, (6, 4))
    fig.add_subplot(3, 1, 1); plt.imshow(cache['blocks.0.attn.hook_q'][0, 0].detach().cpu().numpy())
    fig.add_subplot(3, 1, 2); plt.imshow(cache['blocks.0.attn.hook_q'][0, 1].detach().cpu().numpy())
    fig.add_subplot(3, 1, 3); plt.imshow(cache['blocks.0.attn.hook_q'][0, 2].detach().cpu().numpy())
    plt.show()

In [ ]:
interact(viz_attn_1, x=(0, 113, 1), y=(0, 113, 1))

- Ok I don't know what I was expecting, but not this lol!
- It's BINARY, it's all the same pattern, but flips at wierd intervals - dafuq

In [ ]:
def viz_attn_2(x=0, y=0):
    input_1=t.tensor([[x, y, 113]]).to(device)
    out_1, cache=model.run_with_cache(input_1)    
    fig=plt.figure(0, (6, 4))
    fig.add_subplot(3, 1, 1); plt.imshow(cache['blocks.0.attn.hook_k'][0, 0].detach().cpu().numpy())
    fig.add_subplot(3, 1, 2); plt.imshow(cache['blocks.0.attn.hook_k'][0, 1].detach().cpu().numpy())
    fig.add_subplot(3, 1, 3); plt.imshow(cache['blocks.0.attn.hook_k'][0, 2].detach().cpu().numpy())
    plt.show()

In [ ]:
interact(viz_attn_2, x=(0, 113, 1), y=(0, 113, 1))

Keys change more? Ok, interesting.

In [ ]:
def viz_attn_3(x=0, y=0):
    input_1=t.tensor([[x, y, 113]]).to(device)
    out_1, cache=model.run_with_cache(input_1)    
    fig=plt.figure(0, (6, 4))
    fig.add_subplot(3, 1, 1); plt.imshow(cache['blocks.0.attn.hook_v'][0, 0].detach().cpu().numpy())
    fig.add_subplot(3, 1, 2); plt.imshow(cache['blocks.0.attn.hook_v'][0, 1].detach().cpu().numpy())
    fig.add_subplot(3, 1, 3); plt.imshow(cache['blocks.0.attn.hook_v'][0, 2].detach().cpu().numpy())
    plt.show()

In [ ]:
interact(viz_attn_3, x=(0, 113, 1), y=(0, 113, 1))

Ok, so some VERY specific patterns happenign with keys and queries, and kinda more like what I expect - continuous/random for the values. 

In [ ]:
def viz_4(x=0, y=0):
    input_1=t.tensor([[x, y, 113]]).to(device)
    out_1, cache=model.run_with_cache(input_1)    
    fig=plt.figure(0, (6, 4))
    fig.add_subplot(4, 1, 1); plt.plot(cache['hook_embed'][0, -1, :-1].detach().cpu().numpy())
    fig.add_subplot(4, 1, 2); plt.plot(cache['blocks.0.hook_attn_out'][0, -1, :-1].detach().cpu().numpy())
    fig.add_subplot(4, 1, 3); plt.plot(cache['blocks.0.hook_mlp_out'][0, -1, :-1].detach().cpu().numpy())
    fig.add_subplot(4, 1, 4); plt.plot(cache['blocks.0.hook_resid_post'][0, -1, :-1].detach().cpu().numpy())
    plt.show()

In [ ]:
interact(viz_4, x=(0, 113, 1), y=(0, 113, 1))

In [ ]:
def viz_5(x=0, y=0):
    input_1=t.tensor([[x, y, 113]]).to(device)
    out_1, cache=model.run_with_cache(input_1)    
    fig=plt.figure(0, (6, 4))
    with t.no_grad():
        manual_unembed=model.unembed(cache['hook_embed'].detach())
        fig.add_subplot(4, 1, 1); plt.plot(manual_unembed[0, -1, :-1].detach().cpu().numpy())
        manual_unembed=model.unembed(cache['blocks.0.hook_attn_out'].detach())
        fig.add_subplot(4, 1, 2); plt.plot(manual_unembed[0, -1, :-1].detach().cpu().numpy())
        manual_unembed=model.unembed(cache['blocks.0.hook_mlp_out'].detach())
        fig.add_subplot(4, 1, 3); plt.plot(manual_unembed[0, -1, :-1].detach().cpu().numpy())
        manual_unembed=model.unembed(cache['blocks.0.hook_resid_post'].detach())
        fig.add_subplot(4, 1, 4); plt.plot(manual_unembed[0, -1, :-1].detach().cpu().numpy())
    plt.show()

In [ ]:
interact(viz_5, x=(0, 113, 1), y=(0, 113, 1))

Ok, that's cool.

In [ ]:
for k in cache.keys():
    print(k, cache[k].shape)

In [ ]:
def viz_6(x=0, y=0):
    input_1=t.tensor([[x, y, 113]]).to(device)
    out_1, cache=model.run_with_cache(input_1)    
    fig=plt.figure(0, (6, 4))
    fig.add_subplot(4, 1, 1); plt.plot(cache['blocks.0.mlp.hook_pre'][0, -1, :-1].detach().cpu().numpy())
    fig.add_subplot(4, 1, 2); plt.plot(cache['blocks.0.mlp.hook_post'][0, -1, :-1].detach().cpu().numpy())
    # fig.add_subplot(4, 1, 3); plt.plot(cache['blocks.0.hook_mlp_out'][0, -1, :-1].detach().cpu().numpy())
    # fig.add_subplot(4, 1, 4); plt.plot(cache['blocks.0.hook_resid_post'][0, -1, :-1].detach().cpu().numpy())
    plt.show()

In [ ]:
interact(viz_6, x=(0, 113, 1), y=(0, 113, 1))

In [ ]:
x=60; y=60
input_1=t.tensor([[x, y, 113]]).to(device)
out_1, cache=model.run_with_cache(input_1)

In [ ]:
out_1.shape

In [ ]:
plt.plot(out_1[0, -1, :-1].detach().cpu().numpy())

In [ ]:
plt.plot(cache['blocks.0.hook_resid_post'][0, -1, :-1].detach().cpu().numpy())

- WTF is happening between blocks.0.hook_resid_post and the final output??
- Oh I guess the unembed?

In [ ]:
with t.no_grad():
    manual_unembed=model.unembed(cache['blocks.0.hook_resid_post'].detach())
plt.plot(manual_unembed[0, -1, :-1].detach().cpu().numpy())

In [ ]:
for k in cache.keys():
    print(k, cache[k].shape)

In [ ]:
plt.imshow(cache['blocks.0.attn.hook_q'][0, :,0].detach().cpu().numpy())

In [ ]:
cache['blocks.0.attn.hook_q'][0, :,0]

In [ ]:
plt.imshow(cache['blocks.0.attn.hook_q'][0, :,1].detach().cpu().numpy())

In [ ]:
plt.imshow(cache['blocks.0.attn.hook_q'][0, 0].detach().cpu().numpy())

In [ ]:
plt.imshow(cache['blocks.0.attn.hook_q'][0, 1].detach().cpu().numpy())

Hmm close to binary and inverses, interesting. 

In [ ]:
plt.imshow(cache['blocks.0.attn.hook_k'][0, 0].detach().cpu().numpy())

In [ ]:
plt.imshow(cache['blocks.0.attn.hook_k'][0, 1].detach().cpu().numpy())

In [ ]:
cache['blocks.0.attn.hook_q'][0, 0] #Hmm why are these values so consistent? 

## Now Maybe explore activations across ranges of inputs?

In [ ]:
all_data = t.tensor([(i, j, p) for i in range(p) for j in range(p)]).to(device)
labels = t.tensor([utils.target_fn(i, j) for i, j, _ in all_data]).to(device)
original_logits, cache = model.run_with_cache(all_data)

In [ ]:
# final_pos_resid_initial

## Now explore functional form

In [ ]:
#After I poke around full model will be cool to poke around this functional form!

W_logit = W_out @ W_U #MLP out here 

W_OV = W_V @ W_O #Attn out here
W_neur = W_E @ W_OV @ W_in

W_QK = W_Q @ W_K.transpose(-1, -2)
W_attn = final_pos_resid_initial @ W_QK @ W_E.T / (cfg.d_head**0.5) #This involves some data? That's intersting. 

# Test shapes
assert W_logit.shape == (cfg.d_mlp, cfg.d_vocab - 1)
assert W_neur.shape == (cfg.n_heads, cfg.d_vocab - 1, cfg.d_mlp)
assert W_attn.shape == (cfg.n_heads, cfg.d_vocab - 1)

# Test values
tests.test_effective_weights(W_logit, W_neur, W_attn, model)

In [ ]:
W_logit.shape

In [ ]:
plt.figure(0, (12, 6))
plt.imshow(W_logit.detach().cpu().numpy().T)

Hmm that seems like some interseting structure. 

In [ ]:
plt.plot(W_logit[1, :].detach().cpu().numpy()) #Oh wow yeah very periodic. 

In [ ]:
plt.plot(W_logit[23, :].detach().cpu().numpy()) #Oh wow yeah very periodic. 

Oh wow that one is nice!

In [ ]:
W_OV.shape

In [ ]:
plt.imshow(W_OV[0].detach().cpu().numpy()) #Some structure here for sure, but nothing like mind-blowing. 

In [ ]:
W_neur.shape

In [ ]:
plt.figure(0, (12, 6))
plt.imshow(W_neur[0].detach().cpu().numpy())

In [ ]:
plt.figure(0, (12, 6))
plt.imshow(W_neur[1].detach().cpu().numpy())

In [ ]:
plt.figure(0, (12, 6))
plt.imshow(W_neur[2].detach().cpu().numpy())

In [ ]:
plt.figure(0, (12, 6))
plt.imshow(W_neur[3].detach().cpu().numpy())

In [ ]:
plt.plot(W_neur[3, :, 1].detach().cpu().numpy()) #Oh wow yeah very periodic. 

Ok wow yeah nice

- Ok yeah so functional form is defintely more Fourier-y.
- Interesting interesting.

---

In [ ]:
all_data

In [ ]:
all_data.shape

In [ ]:
labels.shape

In [ ]:
# model

In [ ]:
out,cache=model.run_with_cache(all_data)

In [ ]:
i=3
out[i, -1, :-1].argmax()

In [ ]:
p

In [ ]:
(100+100)%113

In [ ]:
input_1

In [ ]:

# out_1,cache=model.run_with_cache(input_1)
out_1.shape

In [ ]:
out_1[0, -1, :-1].argmax()

In [ ]:
t.nn.Softmax(0)(out_1[0, -1, :-1])

In [ ]:
plt.plot(t.nn.Softmax(0)(out_1[0, -1, :-1]).detach().cpu().numpy())

In [ ]:
#After I poke around full model will be cool to poke around this functional form!

W_logit = W_out @ W_U

W_OV = W_V @ W_O
W_neur = W_E @ W_OV @ W_in

W_QK = W_Q @ W_K.transpose(-1, -2)
W_attn = final_pos_resid_initial @ W_QK @ W_E.T / (cfg.d_head**0.5)

# Test shapes
assert W_logit.shape == (cfg.d_mlp, cfg.d_vocab - 1)
assert W_neur.shape == (cfg.n_heads, cfg.d_vocab - 1, cfg.d_mlp)
assert W_attn.shape == (cfg.n_heads, cfg.d_vocab - 1)

# Test values
tests.test_effective_weights(W_logit, W_neur, W_attn, model)